# 01 — Análise Exploratória de Dados (EDA)

**Projeto:** IoT Guard — Detecção de intrusão em redes IoT com Aprendizado de Máquina Supervisionado

**MBA em Ciência de Dados — Unifor · Turma 12 · Prof. Caio Ponte**

**Base:** RT-IoT2022, do UCI Machine Learning Repository (id 942, DOI 10.24432/C5P338, CC BY 4.0) — tráfego real capturado em uma rede IoT de laboratório com o monitor Zeek e o plugin Flowmeter. Base criada por B. S. Sharmila e Rohini Nagapadma (NIE, Mysuru) e publicada na revista *Cybersecurity* (2023).

**Objetivo deste notebook:**
1. Entender a estrutura, distribuição e desbalanceamento da base.
2. Identificar vazamento de dados (colunas que dão a resposta de graça).
3. Caracterizar as classes por distribuições, protocolo, serviço e outliers.
4. Medir associações (Cramér's V, ANOVA F-score, Mutual Information) e correlações.
5. Reduzir dimensionalidade para visualizar a separabilidade das classes (PCA e t-SNE).
6. Codificar as variáveis categóricas e exportar o dataset limpo para o próximo passo (`02_Preprocessing.ipynb`).

A referência técnica completa do pipeline está nos notebooks:
1. `01_EDA.ipynb` (este) — exploração.
2. `02_Preprocessing.ipynb` — limpeza, escala, balanceamento e split.
3. `03_Model.ipynb` — benchmark e otimização (LightGBM, entre outros).

In [ ]:
# Corrige automaticamente uma instalação incompleta do Matplotlib (ex.: matplotlib.image ausente).
import importlib, shutil, subprocess, sys
from pathlib import Path

import warnings, os
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from pathlib import Path
from scipy.stats import chi2_contingency
from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 12, 'axes.titleweight': 'bold'})

# Caminhos seguindo a estrutura do projeto
PROJECT_ROOT = Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'rt_iot2022'
EDA_DIR      = PROJECT_ROOT / 'artifacts' / 'presentation' / 'eda'
MODEL_DIR    = PROJECT_ROOT / 'artifacts' / 'model' / 'data'

for d in (RAW_DATA_DIR, EDA_DIR, MODEL_DIR):
    d.mkdir(parents=True, exist_ok=True)

TARGET = 'Attack_type'

CORES = {'Normal': '#4C72B0', 'Ataque': '#C44E52'}
NORMAIS = ['Thing_Speak', 'MQTT_Publish', 'Wipro_bulb']
NOMES = {
    'DOS_SYN_Hping': 'DoS SYN', 'Thing_Speak': 'ThingSpeak (normal)',
    'ARP_poisioning': 'ARP poisoning', 'MQTT_Publish': 'MQTT (normal)',
    'NMAP_UDP_SCAN': 'Nmap UDP', 'NMAP_XMAS_TREE_SCAN': 'Nmap XMAS',
    'NMAP_OS_DETECTION': 'Nmap OS', 'NMAP_TCP_scan': 'Nmap TCP',
    'DDOS_Slowloris': 'DDoS Slowloris', 'Wipro_bulb': 'Wipro bulb (normal)',
    'Metasploit_Brute_Force_SSH': 'Brute force SSH', 'NMAP_FIN_SCAN': 'Nmap FIN'
}

def mostrar():
    plt.tight_layout()
    plt.show()

def salvar(nome):
    p = EDA_DIR / f'{nome}.png'
    plt.savefig(p, dpi=120, bbox_inches='tight')
    return p

## 1. Definição do Problema

**Contexto.** Lâmpadas, sensores e assistentes de voz estão em casas, indústrias e hospitais. Esses aparelhos costumam ter pouca memória, firmware simples e senhas fracas — e por isso são alvos fáceis. Um sistema de detecção de intrusão (IDS) observa o tráfego da rede e precisa dizer, para cada conexão, se ela é legítima ou um ataque — e de qual tipo.

**Problema.** Classificar cada fluxo de rede em um de 12 tipos de tráfego: 3 normais (aparelhos IoT legítimos) e 9 ataques.
- Tipo: classificação multiclasse supervisionada.
- Cada linha: um **fluxo de rede**, isto é, uma conexão resumida em estatísticas de pacotes, bytes, tempos e flags TCP.
- Target: `Attack_type`.

| Grupo | Classe | O que é |
|---|---|---|
| Normal | `Thing_Speak` | LED que envia dados para a plataforma ThingSpeak |
| Normal | `MQTT_Publish` | Sensor de temperatura publicando via MQTT |
| Normal | `Wipro_bulb` | Lâmpada inteligente Wipro |
| Ataque | `DOS_SYN_Hping` | Negação de serviço por inundação de pacotes SYN (Hping) |
| Ataque | `DDOS_Slowloris` | Negação de serviço distribuída que mantém conexões HTTP abertas |
| Ataque | `ARP_poisioning` | Envenenamento de ARP (homem no meio) |
| Ataque | `Metasploit_Brute_Force_SSH` | Força bruta de senha SSH (Metasploit) |
| Ataque | `NMAP_UDP_SCAN`, `NMAP_TCP_scan`, `NMAP_XMAS_TREE_SCAN`, `NMAP_FIN_SCAN`, `NMAP_OS_DETECTION` | Varreduras de reconhecimento da rede (Nmap) |

**Motivação.** Saber o tipo de ataque permite reagir do jeito certo: bloquear o IP que tenta adivinhar senhas, limitar pacotes SYN durante um DoS, isolar um aparelho comprometido. Como os aparelhos IoT têm poucos recursos, o modelo também precisa ser **leve** e não pode deixar passar ataques raros.

**Origem dos dados.** RT-IoT2022 (UCI id 942). 123.117 fluxos e 85 colunas originais. O tráfego foi capturado de verdade numa rede IoT de laboratório; o tráfego normal vem de aparelhos reais e os ataques foram executados com ferramentas reais (Hping, Nmap, Metasploit).

## 2. Carregamento com Fallback Local
**O que estamos fazendo:** Verificamos se o dataset já foi salvo localmente (`data/rt_iot2022/RT_IOT2022.csv`). Se sim, carregamos o CSV. Se não, baixamos da UCI e salvamos uma cópia.
**Por que:** Evita perda de tempo se a internet cair ou o servidor da UCI ficar instável. Reproduz o dataset de forma determinística.

In [ ]:
RAW_DATA_PATH = RAW_DATA_DIR / 'RT_IOT2022.csv'

if RAW_DATA_PATH.exists():
    print(f'Carregando dataset do armazenamento local: {RAW_DATA_PATH}')
    df = pd.read_csv(RAW_DATA_PATH)
else:
    print('Baixando dataset da UCI (id=942)...')
    dados = fetch_ucirepo(id=942).data
    if dados is None:
        raise RuntimeError('Não foi possível baixar a RT-IoT2022 da UCI')
    df = dados.original.copy()
    df.to_csv(RAW_DATA_PATH, index=False)
    print(f'Dataset salvo em: {RAW_DATA_PATH}')

print(f'Shape original: {df.shape}')
df.head()

In [ ]:
df.info(verbose=False)

df['Grupo'] = np.where(df[TARGET].isin(NORMAIS), 'Normal', 'Ataque')
print('\nDistribuição por grupo:')
print(df['Grupo'].value_counts().to_string())

### 2.1 Dicionário de dados (por grupo de colunas)

As 83 features descrevem cada fluxo. Como são muitas, o dicionário está agrupado. `fwd` = sentido origem → destino, `bwd` = sentido destino → origem.

| Grupo | Colunas (exemplos) | Descrição | Uso no modelo |
|---|---|---|---|
| Identificação | `id` | Número sequencial da linha **dentro de cada classe** | ❌ removida (vazamento, ver §4) |
| Portas | `id.orig_p`, `id.resp_p` | Porta de origem e de destino | ❌ removidas (artefato do laboratório, ver §4) |
| Protocolo | `proto`, `service` | Protocolo de transporte (tcp/udp/icmp) e serviço de aplicação (dns, mqtt, http…) | ✅ features categóricas |
| Duração e volume | `flow_duration`, `fwd_pkts_tot`, `bwd_pkts_tot`, `fwd_data_pkts_tot` | Duração do fluxo e quantidade de pacotes em cada sentido | ✅ features |
| Taxas | `fwd_pkts_per_sec`, `flow_pkts_per_sec`, `down_up_ratio`, `payload_bytes_per_second` | Velocidade de envio de pacotes e bytes | ✅ features |
| Cabeçalhos | `fwd_header_size_tot/min/max`, `bwd_header_size_*` | Tamanho dos cabeçalhos dos pacotes | ✅ features |
| Flags TCP | `flow_SYN_flag_count`, `flow_ACK_flag_count`, `flow_FIN_flag_count`, `flow_RST_flag_count`… | Quantas vezes cada flag TCP apareceu | ✅ features (`bwd_URG_flag_count` é constante e é removida) |
| Payload | `fwd_pkts_payload.min/max/tot/avg/std`, `bwd_*`, `flow_*` | Estatísticas do conteúdo útil dos pacotes | ✅ features |
| Tempo entre pacotes (IAT) | `fwd_iat.*`, `bwd_iat.*`, `flow_iat.*` | Intervalo entre pacotes consecutivos | ✅ features |
| Subfluxos e rajadas | `fwd_subflow_*`, `fwd_bulk_*`, `bwd_bulk_*` | Comportamento em sub-sequências e rajadas | ✅ features |
| Atividade | `active.*`, `idle.*` | Tempo em que o fluxo ficou ativo ou ocioso | ✅ features |
| Janela TCP | `fwd_init_window_size`, `bwd_init_window_size`, `fwd_last_window_size` | Tamanho da janela de recepção TCP | ✅ features |
| **Target** | `Attack_type` | **Tipo de tráfego (12 classes)** | 🎯 **target** |

## 3. Qualidade e Estatísticas Descritivas
**O que estamos fazendo:** Checamos valores nulos, duplicadas, colunas constantes, valores infinitos e negativos. Depois, olhamos estatísticas descritivas das features-chave.
**Por que:** Antes de qualquer decisão de modelagem é preciso saber o que está faltando ou inconsistente na base. O formato dos dados (muito assimétrico) é o que orienta a escolha de transformação (log) no pré-processamento.

In [ ]:
num_all = df.select_dtypes(include=np.number).drop(columns=['id'], errors='ignore')

print(f'Valores faltantes: {df.isna().sum().sum():,}')
print(f'Valores infinitos (numéricas): {np.isinf(num_all).sum().sum():,}')
print(f'Valores negativos (numéricas): {int((num_all < 0).sum().sum()):,}')
print(f'Linhas duplicadas (ignorando id): {df.drop(columns="id", errors="ignore").duplicated().sum():,}')

constantes = [c for c in df.columns if df[c].nunique() <= 1]
print(f'Colunas constantes: {constantes}')

In [ ]:
CHAVE = ['flow_duration', 'fwd_pkts_tot', 'bwd_pkts_tot',
         'flow_pkts_per_sec', 'fwd_pkts_payload.avg',
         'flow_iat.avg', 'payload_bytes_per_second', 'fwd_init_window_size']

stats = df[CHAVE].describe().T
stats['skew'] = df[CHAVE].skew()
stats['kurtosis'] = df[CHAVE].kurtosis()
display(stats.round(2))

In [ ]:
assimetria = num_all.skew().abs().sort_values(ascending=False)
print(f'Assimetria (skew) mediana das features numéricas: {assimetria.median():.1f}')
print(f'Features com |skew| > 5: {(assimetria > 5).sum()} de {len(assimetria)}')
print('\nColunas categóricas:')
print(df.describe(include=['object']).T.to_string())

**Leitura:**
- Não há valores faltantes nem negativos.
- Há **5.195 linhas duplicadas (4,22%)**. Como o target é muito desbalanceado e algumas classes têm pouquíssimos exemplos, vamos **manter as duplicadas nesta etapa**; a remoção será feita em `02_Preprocessing.ipynb`, **antes do split treino/teste**, para não vazar a mesma linha para os dois conjuntos.
- Existe uma **coluna constante** (`bwd_URG_flag_count`) que sai do dataset.
- As variáveis numéricas são **muito assimétricas**: o skew mediano é ≈27, e a maioria das features tem |skew| > 5. Quase todos os fluxos são curtos e pequenos, e uns poucos são enormes. Isso **justifica o uso de log1p** no pré-processamento (estabiliza a variância e reduz a influência da cauda).

## 4. Balanceamento de Classes
**O que estamos fazendo:** Medimos a distribuição do target em escala logarítmica.
**Por que:** A base é fortemente desbalanceada; sem escala logarítmica as classes minoritárias desaparecem visualmente. Entender esse desbalanceamento é o que **justifica o uso de F1-Macro** como métrica principal (em vez de acurácia).

In [ ]:
cont = df[TARGET].value_counts()

fig, ax = plt.subplots(figsize=(10, 6))
cores = [CORES['Normal'] if c in NORMAIS else CORES['Ataque'] for c in cont.index]
ax.barh([NOMES.get(c, c) for c in cont.index], cont.to_numpy(), color=cores)
ax.set_xscale('log')
for i, v in enumerate(cont.to_numpy()):
    ax.text(v * 1.1, i, f'{v:,} ({v/len(df):.2%})'.replace(',', '.'), va='center', fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Nº de fluxos (escala log)')
ax.set_title('Distribuição de Attack_type — azul = normal, vermelho = ataque')
ax.set_xlim(10, cont.max() * 8)
salvar('distribuicao_target')
mostrar()

print(df[TARGET].value_counts(normalize=True).round(4).to_string())

**Leitura:**
- O desbalanceamento é **extremo**. O `DoS_SYN_Hping` sozinho tem ~77% das linhas. As varreduras FIN e o brute force SSH têm menos de 40 exemplos cada.
- Um modelo que **sempre responde "DoS"** já acerta ~77% — portanto, **acurácia não serve como métrica principal**.
- Vamos usar **F1-Macro** (dá o mesmo peso a todas as classes) e **olhar métricas por classe**. No pré-processamento, testaremos undersampling do DoS e SMOTE nas classes raras.

## 5. Vazamento de Dados: a coluna `id` e as portas
**O que estamos fazendo:** Investigamos duas suspeitas de vazamento (data leakage): a coluna `id` e as colunas de porta `id.orig_p` / `id.resp_p`.
**Por que:** Se uma coluna entrega a resposta sem que o modelo aprenda o comportamento do tráfego, o desempenho no teste será irreal e o modelo falhará em produção contra ataques novos.

In [ ]:
resumo_id = df.groupby(TARGET).agg(linhas=('id', 'size'), id_min=('id', 'min'), id_max=('id', 'max'))
display(resumo_id.sort_values('linhas', ascending=False))

portas = df.groupby(TARGET)['id.resp_p'].agg(lambda s: s.value_counts(normalize=True).iloc[0])
print('\nFração de fluxos na porta de destino mais comum, por classe:')
print(portas.sort_values(ascending=False).round(3).to_string())

In [ ]:
Xa, Xb, ya, yb = train_test_split(
    df[['id', 'id.resp_p']], df[TARGET],
    test_size=0.2, stratify=df[TARGET], random_state=RANDOM_STATE
)
for cols in (['id'], ['id.resp_p']):
    arv = DecisionTreeClassifier(random_state=RANDOM_STATE).fit(Xa[cols], ya)
    acc = accuracy_score(yb, arv.predict(Xb[cols]))
    print(f'Árvore usando só {cols}: acurácia = {acc:.3f}')

**Leitura — duas coisas na base entregam a resposta:**

1. **A coluna `id` não identifica a linha na base inteira: ela recomeça do zero em cada classe.** Um `id` acima de 8.107 (o maior do ThingSpeak, segunda maior classe) só pode ser DoS. Só com essa coluna, uma árvore acerta ~70%. É **vazamento** — a coluna sai.

2. **As portas:** 100% do DoS SYN vai para a porta 21, e o MQTT usa quase sempre a 1883. Só com a porta de destino, a árvore acerta ~91%. Isso é artefato do laboratório (o pesquisador escolheu atacar a porta 21); um atacante de verdade usa qualquer porta. As duas colunas de porta saem, para o modelo ter que aprender pelo **comportamento do tráfego**.

> Confirmamos o mesmo resultado com **Cramér's V** e **Mutual Information** na análise de relevância (§8). `id.resp_p` tem Cramér's V ≈ 0,67 e MI ≈ 0,83. `id.orig_p` tem Cramér's V ≈ 0, mas MI ≈ 0,7 — também permite memorizar. Testes derivados (`resp_known_well` com V ≈ 0,87 e `orig_is_dinamic` com V ≈ 0,45) reforçam o mesmo padrão. **Decisão: remover `id`, `id.orig_p` e `id.resp_p`.**

## 6. Distribuições e Outliers por Classe
**O que estamos fazendo:** Visualizamos a distribuição (em log) de features-chave por classe.
**Por que:** Cada tipo de ataque tem uma assinatura comportamental. Ver as classes separadas ajuda a entender **onde o modelo deve ir bem** e **onde deve errar**.

In [ ]:
ordem = cont.index.tolist()
plot_df = df.assign(Classe=df[TARGET].map(NOMES))

fig, axes = plt.subplots(2, 4, figsize=(18, 10), sharey=True)
for ax, c in zip(axes.flat, CHAVE):
    sns.boxplot(
        data=plot_df.assign(v=np.log1p(plot_df[c])), y='Classe', x='v',
        order=[NOMES[o] for o in ordem], hue='Grupo', palette=CORES,
        dodge=False, legend=False, fliersize=1, ax=ax
    )
    ax.set_title(c, fontsize=10); ax.set_xlabel('log(1 + valor)'); ax.set_ylabel('')
plt.suptitle('Distribuição (em log) das principais features por classe', y=1.01, fontsize=13)
salvar('distribuicao_features_por_classe')
mostrar()

**Leitura:**
- Cada ataque tem um padrão próprio. O **DoS SYN** tem fluxos de 1–2 pacotes, duração quase zero e taxa de pacotes altíssima (é uma inundação). O **Slowloris** tem fluxos longos e lentos (segura conexões abertas). As **varreduras Nmap** são fluxos minúsculos, sem payload.
- O tráfego **normal** (MQTT, ThingSpeak) tem payload e intervalos regulares entre pacotes, como um sensor mandando leituras de tempos em tempos.
- A janela TCP inicial (`fwd_init_window_size`) separa bem as ferramentas de ataque (valores fixos) dos aparelhos reais.
- **ARP poisoning, Slowloris e a lâmpada Wipro** são as classes mais espalhadas e se sobrepõem ao tráfego do ThingSpeak. É ali que o modelo deve errar mais.

In [ ]:
def outliers_iqr(s):
    q1, q3 = float(s.quantile(0.25)), float(s.quantile(0.75))
    iqr = q3 - q1
    return int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())

out = pd.DataFrame({
    'outliers (original)': {c: outliers_iqr(df[c]) for c in CHAVE},
    'outliers (após log1p)': {c: outliers_iqr(pd.Series(np.log1p(df[c]))) for c in CHAVE}
})
out['% das linhas (original)'] = (out.iloc[:, 0] / len(df) * 100).round(1)
display(out)

q1, q3 = float(df['flow_duration'].quantile(0.25)), float(df['flow_duration'].quantile(0.75))
acima = (df['flow_duration'] > q3 + 1.5 * (q3 - q1)).to_numpy()
print('\nClasses dos fluxos outliers em flow_duration (original):')
print(df[TARGET][acima].value_counts().head(6).to_string())

**Leitura — outliers são sinal, não ruído:**
- Pela regra do IQR, em 6 das 8 variáveis **14%–26% das linhas seriam outliers** (nas taxas, nenhuma). Só que, com 77% de DoS minúsculo, os quartis ficam colados no DoS, e **classes inteiras passam a parecer 'atípicas'** (quase todo ThingSpeak e MQTT, o ARP e o Slowloris).
- Esses pontos **são justamente o que diferencia as classes**. Removê-los apagaria a assinatura dos ataques. **Decisão: manter todos os outliers.**
- O `log1p` não muda a contagem (preserva a ordem), mas **encolhe a escala**: um fluxo 1.000 vezes maior fica só umas 7 unidades acima. Isso importa para modelos sensíveis à escala (Regressão Logística, KNN).

## 7. Protocolo e Serviço por Classe
**O que estamos fazendo:** Cruzamos as variáveis categóricas `proto` e `service` com o target.
**Por que:** Se alguma combinação (ex: `service=mqtt`) já identifica a classe, isso é informação útil; se `service=-` (desconhecido) predomina em ataques, isso também é informação.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 6), gridspec_kw={'width_ratios': [1, 3]})
for ax, col in zip(axes, ['proto', 'service']):
    tab = pd.crosstab(df[TARGET].map(NOMES), df[col], normalize='index').reindex([NOMES[o] for o in ordem])
    sns.heatmap(tab, annot=True, fmt='.0%', cmap='Blues', cbar=False, ax=ax, annot_kws={'size': 8})
    ax.set_title(f'{col} dentro de cada classe'); ax.set_xlabel(col); ax.set_ylabel('')
axes[1].set_yticklabels([])
salvar('protocolo_servico_por_classe')
mostrar()

**Leitura:**
- Protocolo e serviço ajudam em **algumas classes**: MQTT usa sempre `mqtt`, Slowloris usa `http` (98%), brute force usa `ssh` (76%), varredura UDP usa `udp` (94%).
- Mas o **DoS e quase todas as varreduras aparecem com `service='-'`** (o Zeek não identificou), e o tráfego normal se divide entre `dns`, `http` e `ssl`.
- **Conclusão:** sozinhas, essas colunas não resolvem — o modelo precisa das **medidas de comportamento** (taxas, tamanhos, IAT). Elas entram como features categóricas, codificadas com LabelEncoder.

## 8. Relevância das Features

### 8.1 Associação das categóricas — Cramér's V
**O que estamos fazendo:** Medimos a força da associação entre `proto`/`service` e o target usando **Cramér's V** (0 = independente, 1 = associação perfeita).
**Por que:** O teste χ² diz se há associação; o Cramér's V dá a **força normalizada**. Isso ajuda a decidir se vale manter a variável.

In [ ]:
def cramers_v(var1, var2):
    tab = pd.crosstab(var1, var2)
    chi2 = chi2_contingency(tab, correction=False)[0]
    n = tab.sum().sum()
    r, k = tab.shape
    phi2 = chi2 / n
    r_corr = max(0, r - (r - 1) ** 2 / (n - 1))
    k_corr = max(0, k - (k - 1) ** 2 / (n - 1))
    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    min_corr = min(k_corr - 1, r_corr - 1)
    return np.sqrt(phi2corr / min_corr) if min_corr != 0 else 0

cat_cols = ['proto', 'service']
df_cramers = pd.DataFrame({
    'feature': cat_cols,
    'cramers_v': [cramers_v(df[c], df[TARGET]) for c in cat_cols],
    'nunique': [df[c].nunique() for c in cat_cols]
}).sort_values('cramers_v', ascending=False)
display(df_cramers)

### 8.2 Importância das numéricas — ANOVA F-score e Mutual Information
**O que estamos fazendo:** Aplicamos **ANOVA F-score** (relações lineares) e **Mutual Information** (lineares e não lineares) entre cada feature numérica e o target.
**Por que:** Duas métricas complementares. A ANOVA é rápida, mas só capta relações lineares; a MI capta qualquer dependência. Combinando as duas, escolhemos features que **realmente discriminam** as classes.

In [ ]:
# Features numéricas válidas (sem target, sem id, sem portas, sem constantes)
num_cols = [c for c in df.select_dtypes(include=np.number).columns
            if c != TARGET]

X_num = df[num_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y_enc = LabelEncoder().fit_transform(df[TARGET])

f_scores, p_values = f_classif(X_num, y_enc)
mi_scores = mutual_info_classif(X_num, y_enc, discrete_features=False, random_state=RANDOM_STATE)

df_scores = (
    pd.DataFrame({'feature': num_cols, 'f_anova': f_scores, 'p_value': p_values, 'mi': mi_scores})
    .sort_values('mi', ascending=False)
    .reset_index(drop=True)
)
display(df_scores.head(20).round(4))

In [ ]:
top_30 = df_scores.head(30).iloc[::-1]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top_30['feature'], top_30['mi'], color='steelblue')
ax.set_title('Top-30 features por Mutual Information')
ax.set_xlabel('Mutual Information')
for c in ax.containers:
    ax.bar_label(c, fmt='%.2f', fontsize=7)
salvar('top30_features_mi')
mostrar()

**Leitura:**
- As features com maior MI estão ligadas ao **formato do fluxo** (tamanhos de pacote, taxas, IAT), e não a identificadores. Isso é o que queremos: comportamento, não atalho.
- `fwd_pkts_payload.min` tem MI alto e correlação **negativa** com o volume total do fluxo (§9). Fluxos grandes quase sempre têm algum pacote vazio, só de controle — é uma informação **diferente** das outras.
- A ANOVA confirma a MI para as relações lineares e dá um `p_value` que permite filtrar features sem significância.

## 9. Matriz de Correlação e Redundância
**O que estamos fazendo:** Calculamos a **correlação de Spearman** (mais robusta a outliers do que Pearson) entre as features numéricas em escala log, mostramos um heatmap resumido e agrupamos colunas com |r| ≥ 0,95.
**Por que:** Colunas redundantes não trazem informação nova, deixam o treino mais lento e dividem a importância de uma medida entre as cópias. O agrupamento + escolha pela MI **reduz dimensionalidade sem perder sinal**.

In [ ]:
REPRES = ['flow_duration', 'fwd_pkts_tot', 'fwd_subflow_pkts', 'fwd_header_size_tot',
          'bwd_pkts_tot', 'flow_pkts_per_sec', 'fwd_pkts_per_sec', 'payload_bytes_per_second',
          'fwd_pkts_payload.min', 'fwd_pkts_payload.avg', 'fwd_pkts_payload.tot',
          'flow_iat.avg', 'flow_iat.max', 'fwd_iat.tot', 'active.avg', 'active.max',
          'idle.avg', 'idle.max', 'flow_SYN_flag_count', 'flow_ACK_flag_count',
          'fwd_init_window_size', 'fwd_last_window_size']

corr = X_num.transform(np.log1p).corr(method='spearman')
corr_rep = corr.loc[REPRES, REPRES]

fig, ax = plt.subplots(figsize=(14, 11.5))
sns.heatmap(
    corr_rep,
    mask=np.triu(np.ones(corr_rep.shape, dtype=bool), k=1),
    annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, annot_kws={'size': 8}, ax=ax,
    cbar_kws={'label': 'Spearman (features em log1p)', 'shrink': 0.8}
)
ax.grid(False)
ax.set_title('Correlação entre 22 features representativas (de 80)')
plt.xticks(rotation=45, ha='right', fontsize=9); plt.yticks(fontsize=9)
salvar('correlacao_spearman')
mostrar()

vals = corr.to_numpy()
i, j = np.triu_indices_from(vals, k=1)
pares = pd.DataFrame({'f1': corr.index[i], 'f2': corr.columns[j], 'r': vals[i, j]})
alto = pares[pares['r'].abs() > 0.95].sort_values('r', ascending=False)
print(f'Pares com |r| > 0.95: {len(alto)}')
print(alto.head(10).round(3).to_string(index=False))

**Leitura:**
- O gráfico mostra **22 das 80 variáveis**, uma ou duas de cada grupo, para ficar legível. A contagem de pares usa as 80.
- Há **muita redundância**. `fwd_pkts_per_sec` × `flow_pkts_per_sec`, `idle.avg` × `idle.max` e `active.avg` × `active.max` têm correlação ≈1,00. `fwd_pkts_tot`, `fwd_subflow_pkts` e `fwd_header_size_tot` ficam entre 0,84 e 0,94.
- O tamanho **mínimo** do payload (`fwd_pkts_payload.min`) anda no sentido **contrário** do volume do fluxo (−0,71 com `fwd_pkts_tot`): fluxos grandes quase sempre têm algum pacote vazio, só de controle. Vai ser uma das features mais importantes.
- **Decisão:** o filtro de correlação entra no `02_Preprocessing.ipynb`, sobre os dados limpos. Não removemos por aqui para evitar remover colunas antes de entender o efeito das duplicadas e do split.

## 10. Redução de Dimensionalidade: PCA e t-SNE
**O que estamos fazendo:** Projetamos as features em 2D para visualizar a separabilidade das classes.
**Por que:**
- O **PCA** (linear) mostra quanta variância está em poucas direções. Se as classes já aparecem separadas no plano, ótimo sinal.
- O **t-SNE** (não linear) preserva vizinhanças locais e revela **ilhas (clusters)** consistentes para cada classe. Isso **antecipa** se modelos baseados em árvore vão conseguir separar as classes.

In [ ]:
# Amostra estratificada: até 300 fluxos por classe, sem duplicatas
amostra = (df.drop_duplicates(subset=df.columns.drop('id', errors='ignore'))
             .sample(frac=1, random_state=RANDOM_STATE)
             .groupby(TARGET).head(300))

Xv = StandardScaler().fit_transform(np.log1p(amostra[num_cols].replace([np.inf, -np.inf], np.nan).fillna(0)))
yv = amostra[TARGET].to_numpy()
print(f'Amostra para projeções: {len(amostra):,} fluxos')

pca = PCA(n_components=2, random_state=RANDOM_STATE).fit(Xv)
Xp = pca.transform(Xv)
Xt = TSNE(n_components=2, perplexity=40, random_state=RANDOM_STATE, init='pca').fit_transform(Xv)
print(f'Variância explicada (2 PCs): {pca.explained_variance_ratio_.sum():.1%}')

In [ ]:
paleta = dict(zip(ordem, sns.color_palette('tab20', len(ordem))))

fig, axes = plt.subplots(1, 2, figsize=(17, 7))
for ax, X2, tit in [
    (axes[0], Xp, f'PCA (variância explicada: {pca.explained_variance_ratio_.sum():.1%})'),
    (axes[1], Xt, 't-SNE')
]:
    for cls in ordem:
        m = yv == cls
        ax.scatter(X2[m, 0], X2[m, 1], s=8, alpha=0.7, color=paleta[cls],
                   label=NOMES[cls], marker='o' if cls not in NORMAIS else 's')
    ax.set_title(tit); ax.set_xlabel('Componente 1'); ax.set_ylabel('Componente 2')
axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left', markerscale=2.5, fontsize=9, title='Classe (■ = normal)')
plt.suptitle(f'Projeções 2D (até 300 fluxos por classe, {len(amostra):,} no total)', y=1.02, fontsize=13)
salvar('pca_tsne')
mostrar()

**Leitura:**
- As duas primeiras componentes do PCA explicam ~60% da variância, e várias classes já aparecem como **faixas separadas** (as varreduras TCP, XMAS e OS, no canto de cima à esquerda).
- No **t-SNE**, quase toda classe forma o seu próprio **grupo**, inclusive as varreduras e o brute force SSH — sinal forte de que as medidas de fluxo carregam bastante informação.
- As **regiões misturadas** são o **ARP poisoning com tráfego normal** (ThingSpeak e Wipro) e alguns fluxos do **Slowloris**. É ali que o modelo deve errar.
- Como o PCA (linear) separa pior que o t-SNE, **um modelo não linear** (Gradient Boosting, Random Forest, LightGBM) tende a ir bem.

## 11. Codificação e Exportação
**O que estamos fazendo:** Codificamos as variáveis categóricas (`proto`, `service`) e o target com `LabelEncoder`. Salvamos também um `target_mapping.csv` para permitir reverter a codificação no Dash.
**Por que:** Modelos baseados em árvore lidam bem com categorias codificadas de forma ordinal. Exportar o dataset limpo evita retrabalho no próximo notebook e **garante reprodutibilidade**.

In [ ]:
df_clean = df.drop(columns=['id', 'id.orig_p', 'id.resp_p'] + constantes, errors='ignore').copy()
df_clean = df_clean.replace([np.inf, -np.inf], np.nan).fillna(0)

# Codificação das categóricas
for col in cat_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

# Codificação do target
le_target = LabelEncoder()
df_clean[TARGET] = le_target.fit_transform(df_clean[TARGET])

# Mapeamento para reverter no Dash
target_mapping = pd.DataFrame({
    'Code': range(len(le_target.classes_)),
    'Attack_Type': le_target.classes_
})
target_mapping.to_csv(MODEL_DIR / 'target_mapping.csv', index=False)
display(target_mapping)

# Exporta dataset limpo e codificado
df_clean.to_csv(MODEL_DIR / 'dataset_encoded.csv', index=False)
print(f'Dataset limpo e codificado salvo em: {MODEL_DIR / "dataset_encoded.csv"}')
print(f'Shape: {df_clean.shape}')

In [ ]:
# Exporta também a lista de top-30 features (por MI) para uso no pré-processamento
top_features = df_scores.head(30)['feature'].tolist()
pd.Series(top_features, name='feature').to_csv(MODEL_DIR / 'top30_features_mi.csv', index=False)
print(f'Top-30 features (por MI) salvas em: {MODEL_DIR / "top30_features_mi.csv"}')
print('\nTop-10:')
for i, f in enumerate(top_features[:10], 1):
    print(f'  {i:2d}. {f}')

## 12. Conclusões

### Sobre os dados
- **Fonte:** RT-IoT2022 — tráfego real de uma rede IoT de laboratório, capturado com Zeek + Flowmeter.
- **Dimensão original:** 123.117 linhas × 85 colunas (81 numéricas + 3 categóricas + `id` + target).
- **Sem valores nulos, sem negativos, sem infinitos** (após tratar `inf`/`-inf` da etapa de limpeza).
- **5.195 linhas duplicadas (4,22%)** — mantidas nesta etapa; a remoção será feita em `02_Preprocessing.ipynb`, **antes do split**, para evitar vazamento treino/teste.
- **1 coluna constante removida** (`bwd_URG_flag_count`).

### Sobre o target (`Attack_type`)
- **Fortemente desbalanceado**, 12 classes: `DOS_SYN_Hping` ≈ 76,5%; as demais vão de ~6,5% para baixo. As varreduras FIN e o brute force SSH têm **menos de 40 exemplos cada**.
- **Conclusão:** acurácia é enganosa (o modelo que sempre responde "DoS" acerta ~77%). A métrica principal do projeto é **F1-Macro**, complementada por métricas por classe. No pré-processamento, testaremos undersampling do DoS e SMOTE nas classes raras.

### Sobre os outliers
- Os dados são **muito assimétricos** (skew mediano ≈ 27). Pela regra do IQR, **14%–26% das linhas** seriam outliers em 6 das 8 features analisadas.
- **Não removemos esses outliers**: no tráfego de rede, valores extremos (fluxo de 1 pacote e 0,00001s, ou um fluxo gigante) **são a assinatura dos ataques** (`SYN_Hping`, Slowloris, varreduras).
- A transformação **log1p** é aplicada no pré-processamento para reduzir a cauda longa sem perder a ordem dos valores.

### Sobre os IDs (`id`, `id.orig_p`, `id.resp_p`)
- **`id`** reinicia em cada classe → árvore simples acerta **~70%** só com ela. **Vazamento.**
- **`id.resp_p`** concentra o DoS SYN na porta 21 → árvore simples acerta **~91%** só com ela. **Artefato do laboratório**, não generaliza. Cramér's V ≈ 0,67 e MI ≈ 0,83 confirmam.
- **`id.orig_p`** tem Cramér's V ≈ 0, mas **MI ≈ 0,7** — também permite memorizar.
- **Decisão:** as três colunas foram **removidas** do dataset final. O modelo aprende pelo **comportamento do tráfego**, não por identificadores.

### Sobre as correlações e a seleção de features
- Existem **muitos pares com |r| ≥ 0,95** (Spearman em log). O agrupamento de colunas redundantes e a escolha pela MI vão acontecer no `02_Preprocessing.ipynb`, **sobre os dados limpos**.
- As features mais relevantes por MI/ANOVA são **comportamentais** (tamanhos, taxas, IAT, payload mínimo) — coerente com o sinal visual do PCA/t-SNE.

### Sobre a separabilidade das classes
- O **t-SNE** mostra ilhas bem definidas para quase todas as classes. As regiões de sobreposição são **ARP poisoning × tráfego normal** e parte do **Slowloris** — onde o modelo deve errar.
- Modelos **não lineares** (Random Forest, Gradient Boosting, LightGBM) tendem a ir melhor que lineares, dado o padrão do t-SNE e a variedade de comportamentos das classes.

### Próximos passos (`02_Preprocessing.ipynb`)
1. Remoção de duplicadas **antes do split** treino/teste.
2. Transformação **log1p** nas features numéricas.
3. **Padronização** (`StandardScaler` ou `RobustScaler`) — apenas onde o modelo exige.
4. **Filtro de correlação** (|r| ≥ 0,95) + seleção pelas top features por MI.
5. **Balanceamento**: undersampling do DoS e/ou SMOTE nas classes raras (dentro do pipeline, para não vazar).
6. Exportação de `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv` para `03_Model.ipynb` e para o Dash.